# 詩織 LoRA v4d テスト生成ノートブック（Kaggle） — LoRA強度比較

v4cへのユーザーの感想（2026-11-19 14:50 JST）：髪はすごく良くなったが、目の丸さと輪郭の子供っぽさが残る。
顔の「形」はベース生成で決まり、顔切り取り仕上げ（v4c）では直しきれなかったため、次はベース生成側の比較テスト。
学習済みLoRA（train-shiori-lora-kaggle-v4 Version 2 の出力 shiori_oneesan_v4.safetensors）を **Add Input** で接続して使う。学習は行わず、推論だけ。

比較条件（すべて v4 と同一：シード42・28ステップ・guidance 7・1024px・同じ2プロンプト）：
- s12：LoRA scale 1.2 のみ（ネガティブは v4 と同じ）→ LoRA側に大人顔の力があるかを見る
- s12n：LoRA scale 1.2 ＋ ネガティブに childish / round-face 系を追加 → 両方効かせた場合を見る
対照は既存の v4 画像（scale 1.0・v4ネガティブ）なので再生成しない。計4枚、実行時間は v4（約3分40秒）の約2倍の見込み。


In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# 依存関係の用意（torchao は古い版が入っていると peft が ImportError になるため外す）
!pip install -q diffusers transformers accelerate peft huggingface_hub
!pip uninstall -y torchao

# ベースモデル（Animagine XL 4.0）を取得
from huggingface_hub import hf_hub_download
import shutil, os
p = hf_hub_download(repo_id='cagliostrolab/animagine-xl-4.0', filename='animagine-xl-4.0.safetensors')
shutil.copy(p, '/kaggle/working/animagine-xl-4.0.safetensors')
print('model ready:', os.path.getsize('/kaggle/working/animagine-xl-4.0.safetensors'))


In [ ]:
# LoRA を読み込んでテスト生成（v4d：LoRA強度 1.2 の比較、計4枚）
import glob, os, torch
from diffusers import StableDiffusionXLPipeline
cands = [c for c in glob.glob('/kaggle/input/**/shiori_oneesan_v4.safetensors', recursive=True)]
print('lora candidates:', cands)
assert cands, 'shiori_oneesan_v4.safetensors not found in /kaggle/input'
lora_path = cands[0]
print('lora size:', os.path.getsize(lora_path))
pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights(os.path.dirname(lora_path), weight_name='shiori_oneesan_v4.safetensors')
# v4 と同じネガティブ（対照条件 s12 用）
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls, 2girls, extra person, tsurime, sharp eyes, narrow eyes'
# s12n 用：子供っぽさ・丸顔を抑える語を追加
neg_n = neg + ', childish face, childlike, round face, young child, chibi, baby face'
os.makedirs('/kaggle/working/output', exist_ok=True)

prompts = [
    ('test_outing_v4d', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, pink cardigan, shopping street, cafe terrace, sunny day'),
    ('test_hair_v4d', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, hair over shoulder, white blouse, indoors, window, daylight, close-up portrait'),
]

# LoRA scale は cross_attention_kwargs で渡す（diffusers の LoRA scale 指定の標準方法。未指定時は 1.0）
for base, prompt in prompts:
    for suffix, neg_prompt in [('s12', neg), ('s12n', neg_n)]:
        g = torch.Generator(device='cuda').manual_seed(42)
        img = pipe(prompt, negative_prompt=neg_prompt, width=1024, height=1024,
                   num_inference_steps=28, guidance_scale=7, generator=g,
                   cross_attention_kwargs={'scale': 1.2}).images[0]
        name = f'{base}_{suffix}.png'
        img.save(f'/kaggle/working/output/{name}')
        print('saved', name)
img
